In [ ]:
# Build the class dataset: all attack rows from the full Parquet + the 5% benign sample CSV.
# Reads from HDFS and writes the combined CSV back to HDFS. Nothing is saved on this machine
# except Spark's temporary working files, which it removes when it finishes.
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .master("local[*]")
         .appName("build_class_dataset")
         .config("spark.driver.memory", "8g")      # lower this if the machine has less RAM
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.session.timeZone", "UTC")

base = ("hdfs://192.168.1.250:9000/tmp/82913-CNT4416-202608/correlated_dataset/"
        "2026-09-26_00-28-45/finalized_data")
out  = "hdfs://192.168.1.250:9000/tmp/82913-CNT4416-202608/class_dataset_csv"

# Benign: the existing 5% sample, every column read as text so nothing is reformatted
benign = spark.read.option("header", True).csv(base + "/Sample_Dataset_CSV/Benign/part-*.csv")
cols = benign.columns                              # column order to keep

# Attacks: every labelled connection from the full dataset (week folders only).
# The Parquet holds benign and attack rows; the filter keeps only attack rows.
full = spark.read.parquet(base + "/Full_Dataset_Parquet/2026-*")
attacks = (full.filter(F.col("label_tactic") != "none")
               .withColumn("datetime", F.date_format("datetime", "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'")))
attacks = attacks.select([(F.col(c).cast("string") if c in attacks.columns
                           else F.lit(None).cast("string")).alias(c) for c in cols])

print("Benign rows (sample):", benign.count())
print("Attack rows (full):  ", attacks.count())
attacks.groupBy("label_tactic").count().orderBy("count", ascending=False).show(truncate=False)

benign.unionByName(attacks).coalesce(1).write.mode("overwrite").option("header", True).csv(out)
print("Written to", out)
